# Qwen3.5-4B-Base — H1 · H3 (Vast / Jupyter)

Запуск **без SSH**: откройте этот ноутбук на Vast Jupyter, выставьте `HF_TOKEN`, выполните ячейки по порядку.

Репо: [niczavadskiy/occupational-gender-bias](https://github.com/niczavadskiy/occupational-gender-bias)  
Датасет: `data/v1/` (45648 = 951×3×2×{2|6}).

In [ ]:
# === config ===
import os
from pathlib import Path

# Вставьте токен HF (нужен, чтобы скачать модель). GitHub-токен НЕ нужен.
os.environ["HF_TOKEN"] = os.environ.get("HF_TOKEN", "hf_PASTE_HERE")
assert not os.environ["HF_TOKEN"].endswith("PASTE_HERE"), "вставьте реальный HF_TOKEN"

from huggingface_hub import login
login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)

MODEL = "Qwen/Qwen3.5-4B-Base"
TAG_PREFIX = "qwen35_4b"

# Корень клона на инстансе (поправьте путь при необходимости)
REPO = Path("/workspace/occupational-gender-bias")
if not (REPO / "src" / "inference.py").is_file():
    REPO = Path.cwd()

RESULTS = Path(os.environ.get("RESULTS_DIR", "/workspace/results"))
RESULTS.mkdir(parents=True, exist_ok=True)
os.environ["RESULTS_DIR"] = str(RESULTS)

os.chdir(REPO)
print("REPO", REPO.resolve())
print("RESULTS", RESULTS)
print("data/v1 shards ok", (REPO / "data/v1/inference_items_v1_man_first.jsonl").is_file())

In [ ]:
# deps (на образе bias-subspaces-env обычно уже есть)
%pip install -q "transformers>=4.50" accelerate huggingface_hub python-dotenv

import torch, transformers
print("torch", torch.__version__, "cuda", torch.cuda.is_available())
print("transformers", transformers.__version__)
if torch.cuda.is_available():
    print("gpu", torch.cuda.get_device_name(0))

In [ ]:
# smoke: токены A/B/C + HS shape 33×2560
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "src.smoke_qwen", "--model", MODEL, "--device", "cuda"])

In [ ]:
# 6 шардов × 7608 (можно запускать по одному / перезапускать упавший)
from pathlib import Path
import subprocess, sys

SHARDS = [
    ("data/v1/inference_items_v1_man_first.jsonl", f"{TAG_PREFIX}_v1_man_first"),
    ("data/v1/inference_items_v1_woman_first.jsonl", f"{TAG_PREFIX}_v1_woman_first"),
    ("data/v1/inference_items_hl_man_man_first.jsonl", f"{TAG_PREFIX}_hl_man_mf"),
    ("data/v1/inference_items_hl_man_woman_first.jsonl", f"{TAG_PREFIX}_hl_man_wf"),
    ("data/v1/inference_items_hl_woman_man_first.jsonl", f"{TAG_PREFIX}_hl_woman_mf"),
    ("data/v1/inference_items_hl_woman_woman_first.jsonl", f"{TAG_PREFIX}_hl_woman_wf"),
]

def already_done(tag: str) -> bool:
    short = MODEL.split("/")[-1]
    hits = sorted(RESULTS.glob(f"run_*_{short}_{tag}"))
    return bool(hits) and (hits[-1] / "per_item.jsonl").is_file()

for items, tag in SHARDS:
    if already_done(tag):
        print("SKIP (exists)", tag)
        continue
    print("===", tag, "===" )
    cmd = [
        sys.executable, "src/inference.py",
        "--model_id", MODEL,
        "--items_file", items,
        "--out_base", str(RESULTS),
        "--run_tag", tag,
    ]
    subprocess.check_call(cmd)
print("all shards done")

In [ ]:
# merge → v1_full_pos_shuffle (15216) + highlight-h3-full (45648)
from datetime import datetime
import subprocess, sys

short = MODEL.split("/")[-1]

def find_run(tag: str) -> Path:
    hits = sorted(RESULTS.glob(f"run_*_{short}_{tag}"))
    assert hits, f"missing run for {tag}"
    return hits[-1]

mf = find_run(f"{TAG_PREFIX}_v1_man_first")
wf = find_run(f"{TAG_PREFIX}_v1_woman_first")
hl_mm = find_run(f"{TAG_PREFIX}_hl_man_mf")
hl_mw = find_run(f"{TAG_PREFIX}_hl_man_wf")
hl_wm = find_run(f"{TAG_PREFIX}_hl_woman_mf")
hl_ww = find_run(f"{TAG_PREFIX}_hl_woman_wf")

out_v1 = RESULTS / f"run_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}_{short}_v1_full_pos_shuffle"
subprocess.check_call([
    sys.executable, "-m", "src.merge_inference_runs",
    "--run-a", str(mf), "--run-b", str(wf),
    "--context-a", "man_first", "--context-b", "woman_first",
    "--run-tag", "v1_full_pos_shuffle",
    "--out", str(out_v1),
])
print("H1 run", out_v1)

out_h3 = RESULTS / "highlight-h3-full"
if out_h3.exists():
    import shutil
    shutil.rmtree(out_h3)
subprocess.check_call([
    sys.executable, "-m", "src.merge_inference_runs",
    "--h3-evidence", "--merge-npz",
    "--run-tag", "h3_evidence_full",
    "--out", str(out_h3),
    "--runs",
    str(out_v1), "no_evidence",
    str(hl_mm), "man",
    str(hl_mw), "man",
    str(hl_wm), "woman",
    str(hl_ww), "woman",
])
print("H3 run", out_h3)

link = RESULTS / "v1_full_pos_shuffle"
if link.exists() or link.is_symlink():
    link.unlink()
try:
    link.symlink_to(out_v1, target_is_directory=True)
except OSError:
    # Windows / no symlink privilege — copy name via text pointer
    link.mkdir(exist_ok=True)
    (link / "SOURCE.txt").write_text(str(out_v1), encoding="utf-8")
print("done")

In [ ]:
# упаковать для скачивания через Jupyter file browser
import tarfile
from datetime import datetime

pack = Path("/workspace") / f"{TAG_PREFIX}_h1h3_pack.tar.gz"
short = MODEL.split("/")[-1]
v1 = sorted(RESULTS.glob(f"run_*_{short}_v1_full_pos_shuffle"))[-1]
h3 = RESULTS / "highlight-h3-full"

with tarfile.open(pack, "w:gz") as tar:
    tar.add(v1, arcname=v1.name)
    tar.add(h3, arcname=h3.name)
print("pack", pack, "MB", round(pack.stat().st_size / 1e6, 1))
print("Скачайте файл из Jupyter file browser и локально:")
print("  python -m src.metrics.h1_v1 --run <v1_full_pos_shuffle>")
print("  python -m src.metrics.h3_v1 --run highlight-h3-full")
print("  python -m probes.v1_rep_run --run <v1> --all   # H11, нужен HS")
print("  python experiments/qwen35-4b-base/analysis/build_combined_report.py")